# Free Kaggle GPU readiness check

This notebook records hardware and package versions only. It does not load a
model, install packages, read project data, train or publish anything.

Use a **private notebook**, select a free GPU accelerator and keep Internet off
for this check. Inspect your account's remaining free quota first. No paid
account linking is needed. GPU access, device type and quota are not guaranteed.
The notebook has not been executed on Kaggle by this project.

Export only this notebook for now. Keep sealed questions/outcomes, credentials,
the project artifact tree and model cache outside it. Any later data export,
training or CUDA generation needs a separate frozen protocol and explicit file
allowlist. Dataset and notebook privacy controls are separate.


In [ ]:
import importlib.metadata as metadata
import json
import platform
from datetime import datetime, timezone
from pathlib import Path

import torch

packages = {}
for name in ("torch", "transformers", "peft", "accelerate", "trl", "bitsandbytes"):
    try:
        packages[name] = metadata.version(name)
    except metadata.PackageNotFoundError:
        packages[name] = None

devices = []
for index in range(torch.cuda.device_count()):
    item = torch.cuda.get_device_properties(index)
    devices.append({"index": index, "name": item.name,
                    "total_memory_bytes": item.total_memory,
                    "compute_capability": [item.major, item.minor]})

report = {
    "version": "kaggle-free-gpu-readiness-v1",
    "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
    "python": platform.python_version(), "platform": platform.platform(),
    "packages": packages, "cuda_available": torch.cuda.is_available(),
    "cuda_runtime": torch.version.cuda, "devices": devices,
    "quota": "Check the account's remaining free quota in the Kaggle UI; not queried here",
    "scope": "Hardware/environment only; no model, dataset, install or training",
    "research_execution_authorized_by_this_notebook": False,
}
print(json.dumps(report, indent=2))
output = Path("/kaggle/working/kaggle-free-gpu-readiness.json")
if output.parent.is_dir():
    with output.open("x") as stream:
        json.dump(report, stream, indent=2, allow_nan=False)
        stream.write("\n")
else:
    print("No Kaggle working directory: report was not saved")
